# Step4: IT 섹터 Black-Litterman 포트폴리오 최적화

Step3에서 생성된 Q, Ω를 B-L view로 사용하여 포트폴리오를 최적화하고 백테스트한다.

**비교 설정 (총 4 전략 + 2 벤치마크):**
| 설정 | 예측 모델 | Prior 방식 |
|------|---------|----------|
| XGB + RP | XGBoost | Risk Parity |
| XGB + MC | XGBoost | Market Cap Weighted |
| TabPFN + RP | TabPFN v2 | Risk Parity |
| TabPFN + MC | TabPFN v2 | Market Cap Weighted |
| 1/N | — | 동일 가중 |
| XLK ETF | — | IT 섹터 ETF |

**고정 파라미터**: γ=3.0, τ=0.025, δ=2.5, W_MAX=0.30, TC=30bps


In [ ]:
"""
Section 0: Import 및 함수 정의
"""
import sys
import platform
import warnings
from pathlib import Path
from math import sqrt
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.optimize as sco
from sklearn.covariance import LedoitWolf

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Windows 한글 출력 인코딩
if sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

warnings.filterwarnings("ignore")

# ─── 한글 폰트 설정 ───────────────────────────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    try:
        import koreanize_matplotlib  # noqa: F401
    except ImportError:
        pass
plt.rcParams["axes.unicode_minus"] = False

# ─── 경로 설정 ────────────────────────────────────────────────────────────────
import os

def _find_base_dir() -> Path:
    # data/panels 디렉터리가 있는 black_litterman 루트를 탐색한다.
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if (candidate / "data" / "panels").is_dir():
            return candidate
    # Fallback: 절대 경로
    return Path("C:/Users/gorhk/최종 프로젝트/finance_project/김재천/black_litterman")

BASE_DIR     = _find_base_dir()
NOTEBOOK_DIR = BASE_DIR / "Test"
DATA_DIR     = BASE_DIR / "data"
OUT_DIR      = NOTEBOOK_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)# ─── 파라미터 ─────────────────────────────────────────────────────────────────
IT_TICKERS   = ["MSFT", "INTC", "ORCL", "AAPL", "CSCO",
                "IBM", "QCOM", "TXN", "CRM", "ADBE"]
N            = len(IT_TICKERS)
DELTA        = 2.5    # 시장 리스크 회피 계수 (균형 수익률 계산)
GAMMA        = 3.0    # MVO 위험 회피 계수 (고정)
TAU          = 0.025  # B-L 사전 분포 불확실성 스케일 (고정)
W_MAX        = 0.30   # 종목별 최대 비중
TC           = 0.003  # 리밸런싱 거래비용 (30bps = 0.30%)
IS_DAYS      = 252
EMBARGO_DAYS = 21
OOS_DAYS     = 21
STEP_SIZE    = 21
DATA_START   = pd.Timestamp("2020-12-01")
DATA_END     = pd.Timestamp("2025-12-31")


# ─── 함수 정의 ────────────────────────────────────────────────────────────────

def load_price_panel(tickers: List[str]) -> pd.DataFrame:
    """
    IT 종목별 adj_close를 합쳐 date × ticker 형태의 가격 패널 반환.
    """
    frames = {}
    for t in tickers:
        df = pd.read_csv(DATA_DIR / "panels" / f"{t}.csv",
                         index_col="date", parse_dates=True)
        df = df[(df.index >= DATA_START) & (df.index <= DATA_END)]
        frames[t] = df["adj_close"]
    return pd.DataFrame(frames).sort_index()


def load_benchmark_xlk() -> pd.Series:
    """XLK ETF adj_close → 일간 수익률 Series."""
    # XLK는 벤치마크 폴더에 저장되어 있음
    path = DATA_DIR / "benchmarks" / "XLK.csv"
    df = pd.read_csv(path, index_col="date", parse_dates=True)
    df = df[(df.index >= DATA_START) & (df.index <= DATA_END)]
    # adj_close 컬럼 찾기 (대소문자 무관)
    ac_col = next((c for c in df.columns if c.lower() in ("adj_close", "adjclose")), df.columns[0])
    return df[ac_col].pct_change().dropna().rename("XLK")


def make_wf_windows(
    dates: pd.DatetimeIndex,
    is_days: int = IS_DAYS,
    embargo: int = EMBARGO_DAYS,
    oos_days: int = OOS_DAYS,
    step: int = STEP_SIZE,
) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    """Walk-forward 윈도우 생성 (Step3와 동일 구조)."""
    windows, n, i = [], len(dates), 0
    while True:
        is_end    = i + is_days
        oos_start = is_end + embargo
        oos_end   = oos_start + oos_days
        if oos_end > n:
            break
        purge = is_end - oos_days
        windows.append((dates[i:purge], dates[oos_start:oos_end]))
        i += step
    return windows


def estimate_cov_ann(returns: pd.DataFrame) -> np.ndarray:
    """
    Ledoit-Wolf 수축 추정기로 IS 일간 수익률의 연환산 공분산 행렬 계산.
    returns: (n_days, n_assets) DataFrame
    """
    lw = LedoitWolf().fit(returns.values)
    return lw.covariance_ * 252   # 연환산


def risk_parity_weights(cov: np.ndarray) -> np.ndarray:
    """
    Equal Risk Contribution (ERC) 가중치 계산.
    목적함수: Σ (w_i * (Σw)_i  - σ²/N)² 최소화
    제약: Σw = 1, 0 ≤ w_i ≤ 1
    """
    n     = cov.shape[0]
    x0    = np.ones(n) / n
    sigma2 = np.diag(cov).mean()  # 목표 위험 기여도 (평균 분산)

    def objective(w):
        risk_contrib = w * (cov @ w)
        return 1e6 * np.sum((risk_contrib - sigma2 / n) ** 2)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, 1.0)] * n
    result = sco.minimize(objective, x0, method="SLSQP",
                          bounds=bounds, constraints=constraints,
                          options={"ftol": 1e-9, "maxiter": 1000})
    w = np.maximum(result.x, 0)
    return w / w.sum()


def mc_weights(tickers: List[str]) -> np.ndarray:
    """
    universe.csv의 2016-01 추정 시총 기반 가중치 계산.
    """
    df_uni = pd.read_csv(DATA_DIR / "universe.csv")
    df_it  = df_uni[df_uni["ticker"].isin(tickers)].set_index("ticker")
    mcap   = df_it.loc[tickers, "mcap_estimate_2016_01"].values.astype(float)
    return mcap / mcap.sum()


def black_litterman(
    Q_vec: np.ndarray,
    Omega_vec: np.ndarray,
    cov: np.ndarray,
    pi: np.ndarray,
    tau: float = TAU,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Black-Litterman 사후 분포 계산 (P = I, 절대 뷰).

    μ_BL = [(τΣ)⁻¹ + Ω⁻¹]⁻¹ [(τΣ)⁻¹π + Ω⁻¹Q]
    Σ_BL = Σ + [(τΣ)⁻¹ + Ω⁻¹]⁻¹

    Q_vec, Omega_vec : (N,) 벡터 (Ω는 대각 요소)
    """
    n           = len(Q_vec)
    Omega_mat   = np.diag(np.maximum(Omega_vec, 1e-8))
    inv_tau_cov = np.linalg.inv(tau * cov)
    inv_omega   = np.linalg.inv(Omega_mat)

    M      = np.linalg.inv(inv_tau_cov + inv_omega)    # (N, N)
    mu_bl  = M @ (inv_tau_cov @ pi + inv_omega @ Q_vec)  # (N,)
    cov_bl = cov + M                                    # (N, N)

    return mu_bl, cov_bl


def mvo_optimize(
    mu: np.ndarray,
    cov: np.ndarray,
    gamma: float = GAMMA,
    w_max: float = W_MAX,
) -> np.ndarray:
    """
    평균-분산 최적화 (MVO).
    max  w'μ - (γ/2) w'Σw
    s.t. Σw = 1, 0 ≤ w_i ≤ w_max
    """
    n   = len(mu)
    x0  = np.ones(n) / n

    def neg_utility(w):
        return -(w @ mu - (gamma / 2) * w @ cov @ w)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds      = [(0.0, w_max)] * n
    result = sco.minimize(neg_utility, x0, method="SLSQP",
                          bounds=bounds, constraints=constraints,
                          options={"ftol": 1e-9, "maxiter": 1000})
    w = np.maximum(result.x, 0)
    return w / w.sum()


def apply_tc(w_new: np.ndarray, w_prev: np.ndarray, tc: float = TC) -> float:
    """리밸런싱 거래비용 = Σ|w_new - w_prev| × tc (round-trip)."""
    return float(np.sum(np.abs(w_new - w_prev)) * tc)


def portfolio_metrics(rets: pd.Series, rf: float = 0.0) -> Dict[str, float]:
    """Ann.Return, Ann.Vol, Sharpe, MDD 계산."""
    if len(rets) == 0:
        return {"ann_return": np.nan, "ann_vol": np.nan, "sharpe": np.nan, "mdd": np.nan}
    ann_ret  = rets.mean() * 252
    ann_vol  = rets.std()  * sqrt(252)
    sharpe   = (ann_ret - rf) / ann_vol if ann_vol > 0 else np.nan
    cum      = (1 + rets).cumprod()
    drawdown = (cum - cum.cummax()) / cum.cummax()
    mdd      = drawdown.min()
    return {"ann_return": ann_ret, "ann_vol": ann_vol, "sharpe": sharpe, "mdd": mdd}


print("Section 0 PASS [OK] — 함수 정의 완료")
print(f"  DELTA={DELTA}, GAMMA={GAMMA}, TAU={TAU}, TC={TC*10000:.0f}bps, W_MAX={W_MAX}")


## Section 1: 데이터 로드 (Q, Ω, 가격 패널, 벤치마크)

In [ ]:
"""
Section 1: Step3 출력 파일 로드 + 가격 패널 + 벤치마크
"""
print("=" * 60)
print("SECTION 1: 데이터 로드")
print("=" * 60)

# ── Step3 Q, Ω 로드 ────────────────────────────────────────────────────────
Q_data, Om_data = {}, {}
for model_name in ["xgb", "tabpfn"]:
    Q_data[model_name]  = pd.read_csv(
        OUT_DIR / f"Q_{model_name}.csv", index_col="date", parse_dates=True
    ).reindex(columns=IT_TICKERS)
    Om_data[model_name] = pd.read_csv(
        OUT_DIR / f"Omega_{model_name}.csv", index_col="date", parse_dates=True
    ).reindex(columns=IT_TICKERS)
    print(f"  [{model_name}] Q shape={Q_data[model_name].shape}, "
          f"날짜범위: {Q_data[model_name].index[0].date()} ~ {Q_data[model_name].index[-1].date()}")

# ── 가격 패널 (일간 수익률 계산용) ─────────────────────────────────────────
price_panel = load_price_panel(IT_TICKERS)
ret_panel   = price_panel.pct_change().dropna()   # date × ticker 일간 수익률
print(f"\n  가격 패널: {price_panel.shape}, 수익률 패널: {ret_panel.shape}")

# ── Walk-forward 윈도우 (Step3와 동일) ─────────────────────────────────────
windows = make_wf_windows(price_panel.index)
print(f"  Walk-forward 윈도우: {len(windows)}개")

# ── XLK 벤치마크 수익률 ────────────────────────────────────────────────────
xlk_ret = load_benchmark_xlk()
print(f"\n  XLK 벤치마크: {len(xlk_ret)}일, "
      f"{xlk_ret.index[0].date()} ~ {xlk_ret.index[-1].date()}")

# ── 시총 가중치 사전 계산 ──────────────────────────────────────────────────
w_mc = mc_weights(IT_TICKERS)
print(f"\n  Market Cap 가중치:")
for t, w in zip(IT_TICKERS, w_mc):
    print(f"    {t:5s}: {w:.4f}")

print("\nSECTION 1 PASS [OK]")


## Section 2: Walk-Forward Black-Litterman 최적화

In [ ]:
"""
Section 2: Walk-Forward B-L 최적화

4개 설정: (XGB|TabPFN) × (Risk Parity|Market Cap)
각 OOS 기간 → 포트폴리오 일간 수익률 기록
"""
print("=" * 60)
print("SECTION 2: Walk-Forward B-L 최적화")
print("=" * 60)

CONFIGS = [
    ("xgb",    "rp",  "XGB + RP"),
    ("xgb",    "mc",  "XGB + MC"),
    ("tabpfn", "rp",  "TabPFN + RP"),
    ("tabpfn", "mc",  "TabPFN + MC"),
]

# 포트폴리오 일간 수익률 누적 딕셔너리
port_rets = {cfg[2]: pd.Series(dtype=float) for cfg in CONFIGS}
port_rets["1/N"] = pd.Series(dtype=float)

# 이전 포트폴리오 비중 (거래비용 계산용)
prev_w = {cfg[2]: np.ones(N) / N for cfg in CONFIGS}
prev_w["1/N"] = np.ones(N) / N

for w_idx, (is_dates, oos_dates) in enumerate(windows):
    oos_first = oos_dates[0].date()
    print(f"  Window {w_idx+1:3d}: IS ~{is_dates[-1].date()} | OOS {oos_first} ~ {oos_dates[-1].date()}")

    # ── IS 수익률 → 공분산 추정 ─────────────────────────────────────────────
    is_rets = ret_panel.loc[ret_panel.index.isin(is_dates), IT_TICKERS].dropna()
    if len(is_rets) < 60:
        print(f"    IS 수익률 행 부족({len(is_rets)}), 스킵")
        continue
    cov_ann = estimate_cov_ann(is_rets)

    # ── OOS 실제 수익률 추출 ────────────────────────────────────────────────
    oos_rets = ret_panel.loc[ret_panel.index.isin(oos_dates), IT_TICKERS]
    if len(oos_rets) == 0:
        continue

    # ── Risk Parity 가중치 ──────────────────────────────────────────────────
    w_rp = risk_parity_weights(cov_ann)

    # ── Q, Ω 추출 (해당 OOS 날짜에서 가장 가까운 예측값 사용) ─────────────
    def get_Q_Omega(model_name: str) -> Tuple[np.ndarray, np.ndarray]:
        """
        OOS 기간과 겹치는 Q, Ω 날짜 찾아 평균 반환.
        없으면 직전 가용 날짜 사용.
        """
        q_df  = Q_data[model_name]
        om_df = Om_data[model_name]
        q_dates = q_df.index

        # OOS 기간과 겹치는 날짜
        overlap = q_dates[q_dates.isin(oos_dates)]
        if len(overlap) == 0:
            # 직전 가용 날짜 fallback
            past = q_dates[q_dates < oos_dates[0]]
            if len(past) == 0:
                return np.zeros(N), np.ones(N) * 1e-4
            overlap = past[[-1]]

        q_vec  = q_df.loc[overlap].mean().fillna(0.0).reindex(IT_TICKERS).values
        om_vec = om_df.loc[overlap].mean().fillna(1e-4).reindex(IT_TICKERS).values
        om_vec = np.maximum(om_vec, 1e-8)
        return q_vec.astype(float), om_vec.astype(float)

    # ── 1/N 포트폴리오 ──────────────────────────────────────────────────────
    w_equal = np.ones(N) / N
    tc_1n   = apply_tc(w_equal, prev_w["1/N"])
    oos_1n  = oos_rets[IT_TICKERS].values @ w_equal
    oos_1n_s = pd.Series(oos_1n, index=oos_rets.index)
    oos_1n_s.iloc[0] -= tc_1n
    port_rets["1/N"] = pd.concat([port_rets["1/N"], oos_1n_s])
    prev_w["1/N"] = w_equal

    # ── B-L 4개 설정 ────────────────────────────────────────────────────────
    for model_name, prior_name, label in CONFIGS:
        q_vec, om_vec = get_Q_Omega(model_name)
        w_prior = w_rp if prior_name == "rp" else w_mc

        # 균형 기대수익률 (prior)
        pi = DELTA * cov_ann @ w_prior

        # B-L 사후 분포
        try:
            mu_bl, cov_bl = black_litterman(q_vec, om_vec, cov_ann, pi, TAU)
            w_opt = mvo_optimize(mu_bl, cov_bl, GAMMA, W_MAX)
        except Exception as e:
            print(f"    [{label}] B-L/MVO 오류: {e}, 1/N fallback")
            w_opt = w_equal

        # OOS 수익률 계산 (거래비용 1일차 차감)
        tc_cost = apply_tc(w_opt, prev_w[label])
        oos_port = oos_rets[IT_TICKERS].values @ w_opt
        oos_s    = pd.Series(oos_port, index=oos_rets.index)
        oos_s.iloc[0] -= tc_cost

        port_rets[label] = pd.concat([port_rets[label], oos_s])
        prev_w[label]    = w_opt

print("\nSECTION 2 PASS [OK]")


## Section 3: 성능 평가 및 시각화

In [ ]:
"""
Section 3: 성능 평가표 + 누적 수익률 그래프
"""
print("=" * 60)
print("SECTION 3: 성능 평가")
print("=" * 60)

# ── XLK 벤치마크 기간 맞추기 ──────────────────────────────────────────────
# 포트폴리오 공통 기간 산출
all_port_idx = pd.concat(list(port_rets.values())).index.unique().sort_values()
xlk_aligned  = xlk_ret.reindex(all_port_idx)

# ── 성능 지표 계산 ─────────────────────────────────────────────────────────
all_strategies = list(port_rets.keys()) + ["XLK"]
perf_rows      = []

for label in all_strategies:
    if label == "XLK":
        rets = xlk_aligned.dropna()
    else:
        rets = port_rets[label].dropna()

    m = portfolio_metrics(rets)
    perf_rows.append({
        "전략"      : label,
        "Ann.Return": f"{m['ann_return']*100:.2f}%",
        "Ann.Vol"   : f"{m['ann_vol']*100:.2f}%",
        "Sharpe"    : f"{m['sharpe']:.3f}",
        "MDD"       : f"{m['mdd']*100:.2f}%",
        "n_days"    : len(rets),
    })

df_perf = pd.DataFrame(perf_rows)
print("\n[성능 요약표]")
print(df_perf.to_string(index=False))

# ── 누적 수익률 시각화 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 색상 구분
colors = {
    "XGB + RP"    : "steelblue",
    "XGB + MC"    : "royalblue",
    "TabPFN + RP" : "tomato",
    "TabPFN + MC" : "firebrick",
    "1/N"         : "gray",
    "XLK"         : "black",
}
ls_map = {
    "XGB + RP"    : "-",
    "XGB + MC"    : "--",
    "TabPFN + RP" : "-",
    "TabPFN + MC" : "--",
    "1/N"         : "-.",
    "XLK"         : ":",
}

ax1 = axes[0]
for label in all_strategies:
    if label == "XLK":
        r = xlk_aligned.dropna()
    else:
        r = port_rets[label].dropna()
    if len(r) == 0:
        continue
    cum = (1 + r).cumprod()
    ax1.plot(cum.index, cum.values,
             label=label, color=colors.get(label, "C0"),
             linestyle=ls_map.get(label, "-"), linewidth=1.5)

ax1.set_title("누적 수익률 비교 (IT 섹터 B-L vs 벤치마크)", fontsize=13)
ax1.set_ylabel("누적 수익률 (원금=1)")
ax1.legend(loc="upper left", fontsize=8)
ax1.grid(True, alpha=0.3)

# 낙폭 (Drawdown)
ax2 = axes[1]
for label in all_strategies:
    if label == "XLK":
        r = xlk_aligned.dropna()
    else:
        r = port_rets[label].dropna()
    if len(r) == 0:
        continue
    cum      = (1 + r).cumprod()
    drawdown = (cum - cum.cummax()) / cum.cummax()
    ax2.plot(drawdown.index, drawdown.values * 100,
             label=label, color=colors.get(label, "C0"),
             linestyle=ls_map.get(label, "-"), linewidth=1.2, alpha=0.8)

ax2.set_title("낙폭 (Drawdown)")
ax2.set_ylabel("Drawdown (%)")
ax2.legend(loc="lower left", fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = OUT_DIR / "bl_performance.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n  그래프 저장: {fig_path}")

# ── 성능 CSV 저장 ──────────────────────────────────────────────────────────
perf_csv = OUT_DIR / "bl_performance.csv"
df_perf.to_csv(perf_csv, index=False, encoding="utf-8-sig")

# 누적 수익률 CSV 저장
cum_frames = {}
for label in all_strategies:
    if label == "XLK":
        r = xlk_aligned.dropna()
    else:
        r = port_rets[label].dropna()
    cum_frames[label] = (1 + r).cumprod() if len(r) > 0 else pd.Series(dtype=float)

cum_df = pd.DataFrame(cum_frames)
cum_csv = OUT_DIR / "bl_cumulative_returns.csv"
cum_df.to_csv(cum_csv, encoding="utf-8-sig")

print(f"  성능 표 저장: {perf_csv}")
print(f"  누적 수익률 저장: {cum_csv}")

print("\nSECTION 3 PASS [OK]")
print("\n" + "=" * 60)
print("Step4 전체 완료")
print("=" * 60)
